# Intermittent requests
The primary mode comparison uses repetition zero and gives each saved model equal weight. The intermittent response latency begins with a prepared input. The 27.61% reduction refers to matched idle windows in the Akida SDK sensor domain.

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/"data/main_fit_metrics.csv").is_file())
OUT = Path(os.environ.get("ARTICLE_RESULTS_DIR", root/"generated")).resolve()
OUT.mkdir(parents=True, exist_ok=True)

def read(name):
    return pd.read_csv(root/"data"/(name+".csv"), float_precision="round_trip")

def save(name, frame):
    frame.to_csv(OUT/(name+".csv"), index=False)

def close(value, expected, tolerance):
    return abs(value-expected) <= tolerance

In [ ]:
sessions = read("intermittent_sessions")
metrics = read("intermittent_batch_metrics")
controls = read("intermittent_cpu_controls")
assert len(sessions) == 78 and sessions.completed_inputs.eq(1).all()
primary = sessions[sessions.repetition.eq(0)]
assert len(primary) == 60 and primary.measurement_id.nunique() == 20
hardware = primary[primary.setting.isin(["lowpower", "performance"])].pivot(index="measurement_id", columns="setting",
    values=["SDK_sensor_idle_W", "SDK_sensor_window_W", "request_latency_ms", "CPU_package_idle_W"])
drop = 100 * (1 - hardware.SDK_sensor_idle_W.lowpower/hardware.SDK_sensor_idle_W.performance)
assert drop.gt(0).all() and hardware.request_latency_ms.lowpower.gt(hardware.request_latency_ms.performance).all()
assert [round(v,2) for v in [drop.min(),drop.median(),drop.max()]] == [27.24,27.61,27.96]
assert round(hardware.request_latency_ms.lowpower.median(),3) == 2.008
assert round(sessions.loc[sessions.setting.eq("lowpower"),"request_latency_ms"].max(),3) == 5.370
assert hardware.CPU_package_idle_W.median().round(2).to_dict() == {"lowpower":8.04,"performance":3.19}
card = []
for setting, frame in primary.groupby("setting"):
    domain = "CPU_package" if setting == "cpu" else "SDK_sensor"
    card.append({"setting": setting, "domain": domain, "n": len(frame),
        "idle_mean_W": frame[f"{domain}_idle_W"].mean(), "idle_median_W": frame[f"{domain}_idle_W"].median(),
        "window_mean_W": frame[f"{domain}_window_W"].mean(), "window_median_W": frame[f"{domain}_window_W"].median(),
        "window_mean_J": frame[f"{domain}_window_J"].mean(), "request_median_ms": frame.request_latency_ms.median(),
        "request_min_ms": frame.request_latency_ms.min(), "request_max_ms": frame.request_latency_ms.max()})
save("intermittent_primary_summary", pd.DataFrame(card))
save("intermittent_paired_idle_reductions", drop.rename("reduction_percent").reset_index())
control = controls.drop(columns=["order","settle_seconds"]).mean()
save("intermittent_cpu_control_means", control.rename("mean_W").rename_axis("quantity").reset_index())
assert control.round(3).tolist() == [2.925,2.951,3.048,2.959,2.964]
a = metrics[metrics.setting.eq("lowpower")].set_index(["order","repetition"]).sort_index().drop(columns="setting")
b = metrics[metrics.setting.eq("performance")].set_index(["order","repetition"]).sort_index().drop(columns="setting")
assert len(a) == len(b) == 26 and a.equals(b)
print(f"Median paired Akida idle power reduction: {drop.median():.2f}%")
display(pd.DataFrame(card), control)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(7.1, 3.5))
for ax, data, unit in [(axes[0],hardware.SDK_sensor_window_W,"SDK sensor request-window power (W)"),
                      (axes[1],hardware.request_latency_ms,"Prepared-input latency (ms)")]:
    for row in data.itertuples():
        ax.plot(["Performance","LowPower"],[row.performance,row.lowpower],marker="o",alpha=.35)
    ax.set_ylabel(unit)
fig.tight_layout()
fig.savefig(OUT/"intermittent_modes.png",dpi=300,bbox_inches="tight")
plt.close(fig)